## Calcolo del modulo quadro dell'ampiezza di decadimento per B⁺ → K⁺π⁺π⁻ℓ⁻ℓ⁺

Questo notebook costruisce l'ampiezza di decadimento per il processo B⁺ → K⁺π⁺π⁻ℓ⁻ℓ⁺ utilizzando il formalismo delle ampiezze di elicità.
L'ampiezza totale è costruita sommando i contributi delle diverse risonanze intermedie (canali three-body, one-resonance e two-resonances) e del termine non risonante.
Il modulo quadro, sommato sulle chiralità della corrente leptonica nel limite di leptoni senza massa, viene espanso lasciando la dipendenza angolare nelle matrici di Wigner e salvato in `WIGAtotsq_terms*.pkl`.

La decomposizione angolare (base ortogonale di funzioni di Wigner reali e momenti associati) è svolta in `MomentiReIm.ipynb`, che legge il file salvato qui.

### Setup iniziale
Importa le librerie necessarie e definisce il valore massimo dei momenti angolari considerati.

In [1]:
import winsound
from tqdm.auto import tqdm
import sympy as sp
from sympy.physics.wigner import clebsch_gordan
from sympy.physics.quantum.spin import Rotation as rot
from sympy import pi
from random import randint
import psutil
import dill
import os

sp.init_printing(use_unicode=True, wrap_line=True, num_columns=160)

import time
start_time = time.time()

Jmax = 2
Lmax = 2

## Definizione dei simboli e funzioni
Definisce tutti i simboli matematici, le funzioni (ampiezze di elicità, propagatori, Blatt-Weisskopf) e i coefficienti necessari per costruire l'ampiezza.

In [2]:
# --- Definizione simboli ---
J, LB, LR, L1, L2, JX, lam, lam1, lam2, eta = sp.symbols('J LB LR L1 L2 JX lam lam1 lam2 eta', integer=True)
lambda1, lambda2 = sp.symbols('lambda1 lambda2', real=True)

# ---  Variabili angolari ---
theta1, theta2, thetal, phi, chi = sp.symbols('theta1 theta2 thetal phi chi', real=True)

# --- Chiralità della corrente leptonica ---
eta_par = sp.symbols('eta_par', integer=True)

# --- Simbolo per la risonanza generica ---
Res = sp.symbols('Res')

# --- Helicity amplitudes e altri coefficienti simbolici ---
H = sp.Function('H', complex=True)(eta, J, lam, LB, Res)     # Helicity amplitude per B -> K_J
Hbar = sp.Function('Hbar', complex=True)(eta, J, lam, LB, Res)     # Helicity amplitude per B -> K*_J
G = sp.Function('G', complex=True)(eta, J, lam, LB, Res)  # Helicity Amplitude for K_J -> R P1
LHA = sp.Function('LHA', real=True)(eta_par, lam1, lam2)  # Leptonic Helicity amplitude

# --- Propagatore ---
Prop = sp.Function('R')(Res)

# --- Blatt-Weisskopf factor ---
L, rBW = sp.symbols('L rBW', positive=True, real=True)
rB, r3, r2 = sp.symbols('rB r3 r2', positive=True, real=True)
q = sp.symbols('q', real=True)
qR = sp.symbols('qR', real=True)

# funzione simbolica generica
BWfunc = sp.Function('BW', real=True)
# istanza base (con argomenti simbolici)
BW = BWfunc(L, rBW, qR)

# --- Insieme dei coefficienti per le ampiezze intermedie ---
atot = []
a_NR = sp.symbols('a_NR', complex=True) # contributo non-risonante
atot.append(a_NR)  

## Costruzione dell'ampiezza A_three-body per il decadimento B⁺ → (R⁺ → K⁺π⁺π⁻)ℓ⁻ℓ⁺
Calcola l'ampiezza per il canale con la risonanza three-body K_J.
Include risonanze K(1460), K1(1650), K2(1820).

In [3]:
### Cella per costruire l'ampiezza A3 ###

resonances3 = {
    #(R, J (spin della risonanza), LR (mom. ang. orb. tra P1 e (P2P3)), JX (momento angolare orbitale tra P2 e P3)), a_i (coefficiente complesso associato alla singola ampiezza di decadimento))
    ('K(1460)', 0, 0, 0): sp.symbols(f'a1', complex=True), #1
    ('K1(1650)', 1, 0, 1): sp.symbols(f'a2', complex=True), #2
    ('K1(1650)', 1, 1, 0): sp.symbols(f'a3', complex=True), #3
    ('K2(1820)', 2, 0, 2): sp.symbols(f'a4', complex=True), #4
    ('K2(1820)', 2, 2, 0): sp.symbols(f'a5', complex=True), #5
}


a3 = []
for a in resonances3.values():
    a3.append(a)
atot.extend(a3)

# --- Costruzione dell'ampiezza A3 ---
A3 = 0  # Inizializza l'ampiezza

for J_val in range(0, Jmax + 1):
    for LB_val in range(abs(J_val-1), abs(J_val-1) + 1):
        print(f"\n\nCalcolando contributi per J = {J_val}, LB = {LB_val}")
        # Somma su tutte le risonanze R con spin = J_val
        for idx_res, keys in enumerate(resonances3.items()):
            R_name, spin, LR, JX = keys[0] # Estrai i valori dalla tupla corrispondenti ai canali selezionati
            coeff_a = keys[1]
            if spin == J_val:
                R_symbol = sp.Symbol(R_name)
                for JX_val in [JX]:
                    for lamx_val in range(-JX_val, JX_val + 1):
                        for LR_val in range(abs(J_val - JX_val), abs(J_val - JX_val) + 1):
                            for lam_val in range(-min(1,J_val), min(1,J_val) + 1):
                                H_term = H.subs(J, J_val).subs(LB, LB_val).subs(lam, lam_val).subs(Res, R_symbol)
                                # usa il coefficiente corrispondente dalla lista a3
                                term = (
                                    coeff_a * H_term * clebsch_gordan(J_val, 1, LB_val, lam_val, -lam_val, 0) *
                                    LHA.subs(eta_par, eta).subs(lam1, lambda1).subs(lam2, lambda2) *
                                    sp.conjugate(rot.D(1, lam_val, lambda1 - lambda2, 0, thetal, 0)) * 
                                    sp.sqrt(sp.Rational((2*LR_val + 1),(2*J_val + 1))) * 
                                    sp.sqrt(sp.Rational((2*J_val + 1),4)/pi) * sp.sqrt(sp.Rational((2*JX_val + 1),4)/pi) * 
                                    clebsch_gordan(LR_val, JX_val, J_val, lam_val - lamx_val, lamx_val, lam_val) *
                                    sp.conjugate(rot.D(J_val, lam_val, lamx_val, phi, theta1, 0)) *
                                    sp.conjugate(rot.D(JX_val, lamx_val, 0, chi, theta2, 0)) *
                                    Prop.subs(Res, R_symbol) *
                                    BW.subs(L, LB_val).subs(rBW, rB).subs(qR, sp.Symbol('B')) * BW.subs(L, J_val).subs(rBW, r3).subs(qR, R_symbol)
                                    )
                                A3 += term
                                print(f"\nR = {R_name}, (spin = {spin})")
                                print(f"Processando {R_name} (spin {J_val}) lambda = {lam_val}, LB = {LB_val}")
                                print(f"JX = {JX_val}, lambdaX = {lamx_val}, LR = {LR_val}")
                                print(f"term = {term}")

# Mostra l'ampiezza totale
print("\n" + "="*50)
print("Ampiezza A3 totale:")
if A3 == 0:
    print("L'ampiezza A3 è nulla (nessuna risonanza selezionata).")
else:
    print(f"Termini nell'ampiezza A3: {len(A3.as_ordered_terms())}")
print("="*50)

# Mostra l'ampiezza
sp.pprint(A3)



Calcolando contributi per J = 0, LB = 1

R = K(1460), (spin = 0)
Processando K(1460) (spin 0) lambda = 0, LB = 1
JX = 0, lambdaX = 0, LR = 0
term = a1*BW(0, r3, K(1460))*BW(1, rB, B)*H(eta, 0, 0, 1, K(1460))*LHA(eta, lambda1, lambda2)*R(K(1460))*conjugate(WignerD(0, 0, 0, chi, theta2, 0))*conjugate(WignerD(0, 0, 0, phi, theta1, 0))*conjugate(WignerD(1, 0, lambda1 - lambda2, 0, thetal, 0))/(4*pi)


Calcolando contributi per J = 1, LB = 0

R = K1(1650), (spin = 1)
Processando K1(1650) (spin 1) lambda = -1, LB = 0
JX = 1, lambdaX = -1, LR = 0
term = a2*BW(0, rB, B)*BW(1, r3, K1(1650))*H(eta, 1, -1, 0, K1(1650))*LHA(eta, lambda1, lambda2)*R(K1(1650))*conjugate(WignerD(1, -1, -1, phi, theta1, 0))*conjugate(WignerD(1, -1, 0, chi, theta2, 0))*conjugate(WignerD(1, -1, lambda1 - lambda2, 0, thetal, 0))/(4*pi)

R = K1(1650), (spin = 1)
Processando K1(1650) (spin 1) lambda = 0, LB = 0
JX = 1, lambdaX = -1, LR = 0
term = 0

R = K1(1650), (spin = 1)
Processando K1(1650) (spin 1) lambda = 1, LB = 

## Costruzione dell'ampiezza A_one-res per il decadimento B⁺ → P₁⁺(R⁰ → P₂⁺π⁻) ℓ⁻ℓ⁺
Calcola l'ampiezza per il canale con la risonanza K_J* che decade in P₂ e P₃.

In [4]:
### Cella per costruire l'ampiezza AR ###

# Lista delle risonanze come tuple (nome, spin)
resonancesR = {
    # (R, J: spin della risonanza, L1, L2), dove L1 è il mom. ang. orb. tra P1 e R, R2 è uguale allo spin di R.
    # a_i: Coefficiente complesso associato alla singola ampiezza di decadimento
    ('rho(770)', 1, 0, 1): sp.symbols(f'a6', complex=True), #6
    ('omega(782)', 1, 0, 1): sp.symbols(f'a7', complex=True), #7
    ('phi(1020)', 1, 0, 1): sp.symbols(f'a8', complex=True), #8
    ('f2(1270)', 2, 0, 2): sp.symbols(f'a9', complex=True), #9
    ('f0(1500)', 0, 0, 0): sp.symbols(f'a10', complex=True), #10
    ('K*(892)', 1, 0, 1): sp.symbols(f'a11', complex=True), #11
    ('K*(1410)', 1, 0, 1): sp.symbols(f'a12', complex=True), #12
    ('K0*(1430)', 0, 0, 0): sp.symbols(f'a13', complex=True), #13
    ('K2*(1430)', 2, 0, 2): sp.symbols(f'a14', complex=True), #14
    ('K*(1680)', 1, 0, 1): sp.symbols(f'a15', complex=True), #15
    ('K0*(1950)', 0, 0, 0): sp.symbols(f'a16', complex=True), #16
}

aR = []
for a in resonancesR.values():
    aR.append(a)
atot.extend(aR)

# --- Costruzione dell'ampiezza AR ---
AR = 0
for J_val in range(0, Jmax + 1):
    for LB_val in range(abs(J_val-1), abs(J_val-1) + 1):
        print(f"\n\nCalcolando contributi per J = {J_val}, LB = {LB_val}")
        for idx_res, keys in enumerate(resonancesR.items()):
            R_name, spin, L1, L2 = keys[0]
            coeff_a = keys[1]
            if spin == J_val:
                R_symbol = sp.Symbol(R_name)    
                for lam_val in range(-min(1,J_val), min(1,J_val) + 1):
                    G_term = G.subs(J, J_val).subs(LB, LB_val).subs(lam, lam_val).subs(Res, R_symbol)
                    coeff_a = aR[idx_res]
                    term = (
                        coeff_a * G_term * clebsch_gordan(J_val, 1, LB_val, lam_val, -lam_val, 0) *
                        LHA.subs(eta_par, eta).subs(lam1, lambda1).subs(lam2, lambda2) *
                        sp.conjugate(rot.D(1, lam_val, lambda1 - lambda2, 0, thetal, 0)) * 
                        sp.sqrt(sp.Rational(1,4)/pi) * sp.sqrt(sp.Rational((2*J_val + 1),4)/pi) * 
                        sp.conjugate(rot.D(J_val, lam_val, lam_val, phi, theta1, 0)) *
                        sp.conjugate(rot.D(J_val, lam_val, 0, chi, theta2, 0)) *
                        Prop.subs(Res, R_symbol) *
                        BW.subs(L, LB_val).subs(rBW, rB).subs(qR, sp.Symbol('B')) * BW.subs(L, J_val).subs(rBW, r3).subs(qR, R_symbol)
                        )
                    AR += term
                    print(f"\nR = {R_name}, (spin = {spin})")
                    print(f"Processando {R_name} (spin {J_val}) lambda = {lam_val}, LB = {LB_val}")
                    print(f"term = {term}")


# Mostra l'ampiezza totale
print("\n" + "="*50)
if AR == 0:
    print("L'ampiezza AR è nulla (nessuna risonanza selezionata).")
else:
    print("Ampiezza AR totale:")
    print(f"Termini nell'ampiezza AR: {len(AR.as_ordered_terms())}")
print("="*50)

# Mostra l'ampiezza
sp.pprint(AR)



Calcolando contributi per J = 0, LB = 1

R = f0(1500), (spin = 0)
Processando f0(1500) (spin 0) lambda = 0, LB = 1
term = a10*BW(0, r3, f0(1500))*BW(1, rB, B)*G(eta, 0, 0, 1, f0(1500))*LHA(eta, lambda1, lambda2)*R(f0(1500))*conjugate(WignerD(0, 0, 0, chi, theta2, 0))*conjugate(WignerD(0, 0, 0, phi, theta1, 0))*conjugate(WignerD(1, 0, lambda1 - lambda2, 0, thetal, 0))/(4*pi)

R = K0*(1430), (spin = 0)
Processando K0*(1430) (spin 0) lambda = 0, LB = 1
term = a13*BW(0, r3, K0*(1430))*BW(1, rB, B)*G(eta, 0, 0, 1, K0*(1430))*LHA(eta, lambda1, lambda2)*R(K0*(1430))*conjugate(WignerD(0, 0, 0, chi, theta2, 0))*conjugate(WignerD(0, 0, 0, phi, theta1, 0))*conjugate(WignerD(1, 0, lambda1 - lambda2, 0, thetal, 0))/(4*pi)

R = K0*(1950), (spin = 0)
Processando K0*(1950) (spin 0) lambda = 0, LB = 1
term = a16*BW(0, r3, K0*(1950))*BW(1, rB, B)*G(eta, 0, 0, 1, K0*(1950))*LHA(eta, lambda1, lambda2)*R(K0*(1950))*conjugate(WignerD(0, 0, 0, chi, theta2, 0))*conjugate(WignerD(0, 0, 0, phi, theta1, 0))*co

## Costruzione dell'ampiezza A_two-res per il decadimento B⁺ → [R₁⁺ → P₁⁺(R₂⁰ → P₂⁺π⁻)] ℓ⁻ℓ⁺
Calcola l'ampiezza per il canale con due risonanze.

In [5]:
### Cella per costruire l'ampiezza AR1R2 ###

resonancesR1R2 = {
    # (R1, J1: spin della risonanza 1, R2, J2: spin della risonanza 2, R1_type (K o K*), L1, L2), dove L1 è il mom. ang. orb. tra P1 e R1, L2 è il mom. ang. orb. tra P2 e R2.
    # a_i: Coefficiente complesso associato alla singola ampiezza di decadimento
    ('K(1460)', 0, 'rho(770)', 1, 'K', 1, 1): sp.symbols(f'a17', complex=True), #17
    ('K(1460)', 0, 'K*(892)', 1, 'K', 1, 1): sp.symbols(f'a18', complex=True), #18
    ('K1(1270)', 1, 'rho(770)', 1, 'K', 0, 1): sp.symbols(f'a19', complex=True), #19
    ('K1(1270)', 1, 'omega(782)', 1, 'K', 0, 1): sp.symbols(f'a20', complex=True), #20
    ('K1(1270)', 1, 'K*(892)', 1, 'K', 0, 1): sp.symbols(f'a21', complex=True), #21
    ('K1(1400)', 1, 'K*(892)', 1, 'K', 0, 1): sp.symbols(f'a22', complex=True), #22
    ('K*(1410)', 1, 'K*(892)', 1, 'K*', 1, 1): sp.symbols(f'a23', complex=True), #23
    ('K*(1680)', 1, 'rho(770)', 1, 'K*', 1, 1): sp.symbols(f'a24', complex=True), #24
    ('K*(1680)', 1, 'K*(892)', 1, 'K*', 1, 1): sp.symbols(f'a25', complex=True), #25
    ('K2*(1430)', 2, 'rho(770)', 1, 'K*', 2, 1): sp.symbols(f'a26', complex=True), #26
    ('K2*(1430)', 2, 'omega(782)', 1, 'K*', 2, 1): sp.symbols(f'a27', complex=True), #27
    ('K2*(1430)', 2, 'K*(892)', 1, 'K*', 2, 1): sp.symbols(f'a28', complex=True), #28
    ('K2(1700)', 2, 'K2*(1430)', 2, 'K', 0, 2): sp.symbols(f'a29', complex=True), #29
    ('K2(1700)', 2, 'K*(892)', 1, 'K', 1, 1): sp.symbols(f'a30', complex=True), #30
    ('K2(1820)', 2, 'K2*(1430)', 2, 'K', 0, 2): sp.symbols(f'a31', complex=True), #31
    ('K2(1820)', 2, 'phi(1020)', 1, 'K', 1, 1): sp.symbols(f'a32', complex=True), #32
    ('K2*(1980)', 2, 'phi(1020)', 1, 'K*', 2, 1): sp.symbols(f'a33', complex=True), #33
}



# Dizionario dei coefficienti a(R1,R2) per ogni coppia di risonanze R1 e R2
aR1R2 = []
for a in resonancesR1R2.values():
    aR1R2.append(a)
atot.extend(aR1R2)

# --- Costruzione dell'ampiezza AR1R2 ---
AR1R2 = 0
for J1_val in range (0, Jmax + 1):
    for LB_val in range(abs(J1_val-1), abs(J1_val-1) + 1):
        print(f"\n\nCalcolando contributi per J1 = {J1_val}, LB = {LB_val}")
        for key, coeff_a in resonancesR1R2.items():
            R1_name, R1_spin, R2_name, R2_spin, R1_type, L1, L2 = key
            if R1_spin == J1_val:  # Solo se lo spin corrisponde a J1_val
                if R1_type == 'K': auxH = H
                if R1_type == 'K*': auxH = Hbar
                R1_symbol = sp.Symbol(R1_name)
                R2_symbol = sp.Symbol(R2_name)
                J2_val = L2  # Considera solo J2_val = L2
                if R2_spin == J2_val:
                    for L1_val in [L1]:
                        for lam_val in range(-min(1,J1_val), min(1,J1_val) + 1):
                            Hterm = auxH.subs(J, J1_val).subs(LB, LB_val).subs(lam, lam_val).subs(Res, R1_symbol)
                            for lam2_val in range(-min(1,J2_val), min(1,J2_val) + 1):
                                term = (
                                    coeff_a * Hterm * clebsch_gordan(J1_val, 1, LB_val, lam_val, -lam_val, 0) *
                                        LHA.subs(eta_par, eta).subs(lam1, lambda1).subs(lam2, lambda2) *
                                        sp.conjugate(rot.D(1, lam_val, lambda1 - lambda2, 0, thetal, 0)) *
                                        sp.sqrt(sp.Rational((2*J1_val + 1),4)/pi) * sp.sqrt(sp.Rational((2*J2_val + 1),4)/pi) * 
                                        sp.sqrt(sp.Rational((2*L1_val + 1),(2*J1_val + 1))) * 
                                        clebsch_gordan(L1_val, J2_val, J1_val, lam_val-lam2_val, lam2_val, lam_val) *
                                        sp.conjugate(rot.D(J1_val, lam_val, lam2_val, phi, theta1, 0)) *
                                        sp.conjugate(rot.D(J2_val, lam2_val, 0, chi, theta2, 0)) *
                                        Prop.subs(Res, R1_symbol) * Prop.subs(Res, R2_symbol) *
                                        BW.subs(L, LB_val).subs(rBW, rB).subs(qR, sp.Symbol('B')) *
                                        BW.subs(L, L1_val).subs(rBW, r2).subs(qR, R1_symbol) * 
                                        BW.subs(L, J2_val).subs(rBW, r2).subs(qR, R2_symbol)
                                        )
                                AR1R2 += term
                                print(f"\nR1 = {R1_name}, (spin = {R1_spin}), R2 = {R2_name}, (spin = {R2_spin}), lamda = {lam_val}, lam2 = {lam2_val}")
                                print(f"LB = {LB_val}, L1 = {L1}, L2 = {L2}")
                                print(f"term = {term}")

# Mostra l'ampiezza totale
print("\n" + "="*50)
print("Ampiezza AR1R2 totale:")
if AR1R2 == 0:
    print("L'ampiezza AR1R2 è nulla (nessuna risonanza selezionata).")
else:
    print(f"Termini nell'ampiezza AR1R2: {len(AR1R2.as_ordered_terms())}")
print("="*50)



Calcolando contributi per J1 = 0, LB = 1

R1 = K(1460), (spin = 0), R2 = rho(770), (spin = 1), lamda = 0, lam2 = -1
LB = 1, L1 = 1, L2 = 1
term = sqrt(3)*a17*BW(1, r2, K(1460))*BW(1, r2, rho(770))*BW(1, rB, B)*H(eta, 0, 0, 1, K(1460))*LHA(eta, lambda1, lambda2)*R(K(1460))*R(rho(770))*conjugate(WignerD(0, 0, -1, phi, theta1, 0))*conjugate(WignerD(1, -1, 0, chi, theta2, 0))*conjugate(WignerD(1, 0, lambda1 - lambda2, 0, thetal, 0))/(4*pi)

R1 = K(1460), (spin = 0), R2 = rho(770), (spin = 1), lamda = 0, lam2 = 0
LB = 1, L1 = 1, L2 = 1
term = -sqrt(3)*a17*BW(1, r2, K(1460))*BW(1, r2, rho(770))*BW(1, rB, B)*H(eta, 0, 0, 1, K(1460))*LHA(eta, lambda1, lambda2)*R(K(1460))*R(rho(770))*conjugate(WignerD(0, 0, 0, phi, theta1, 0))*conjugate(WignerD(1, 0, 0, chi, theta2, 0))*conjugate(WignerD(1, 0, lambda1 - lambda2, 0, thetal, 0))/(4*pi)

R1 = K(1460), (spin = 0), R2 = rho(770), (spin = 1), lamda = 0, lam2 = 1
LB = 1, L1 = 1, L2 = 1
term = sqrt(3)*a17*BW(1, r2, K(1460))*BW(1, r2, rho(770))*BW(1, 

## Definizione delle ampiezze di elicità leptoniche nel limite massless
Definisce le ampiezze di elicità leptoniche (LHA), considerando le chiralità ±1 della corrente leptonica, (V ± A).

In [6]:
b = sp.symbols('b', real = True) # beta_ell = sqrt(1 - 4 m_l^2 / q^2)
ml = sp.symbols('m_l', real = True) # massa del leptone

Leptonic_subs = {
    ###Leptonic Helicity Amplitudes MASSLESS
    LHA.subs({eta_par: -1, lam1: sp.Rational(-1,2), lam2: sp.Rational(-1,2)}): 0,
    LHA.subs({eta_par: -1, lam1: sp.Rational(-1,2), lam2: sp.Rational(1,2)}): (2)*sp.sqrt(2)*q,
    LHA.subs({eta_par: -1, lam1: sp.Rational(1,2), lam2: sp.Rational(-1,2)}): 0,
    LHA.subs({eta_par: -1, lam1: sp.Rational(1,2), lam2: sp.Rational(1,2)}): 0,

    LHA.subs({eta_par: 1, lam1: sp.Rational(-1,2), lam2: sp.Rational(-1,2)}): 0,
    LHA.subs({eta_par: 1, lam1: sp.Rational(-1,2), lam2: sp.Rational(1,2)}): 0,
    LHA.subs({eta_par: 1, lam1: sp.Rational(1,2), lam2: sp.Rational(-1,2)}): (2)*sp.sqrt(2)*q,    
    LHA.subs({eta_par: 1, lam1: sp.Rational(1,2), lam2: sp.Rational(1,2)}): 0,
}

## Calcolo dell'ampiezza totale al quadrato
Somma tutti i contributi Atot = non-risonante + A(three-body) + A(one-res) + A(two-res), calcola il modulo quadrato.
Non espande le funzioni di Wigner, in modo che la dipendenza dalle variabili angolari sia tutta nelle D.
Salva il risultato in file .pkl.

In [7]:
# AMPIEZZA TOTALE

print("Verificando la presenza di file salvati...")

# Determina il nome del file
atotsq_file = None

# Cerca i file .pkl nella directory corrente
for file in os.listdir(os.getcwd()):
    if file.startswith("WIGAtotsq_terms") and file.endswith(".pkl"):
        atotsq_file = file

# Controlla se entrambi i file esistono
if atotsq_file and os.path.exists(atotsq_file):
    print("Founded files. Import of saved data.")
    
    try:
        # Carica Atotsq
        with open(atotsq_file, 'rb') as f:
            Atotsqfile = dill.load(f)
            Atotsq = Atotsqfile['Atotsq']
            terms_list = Atotsqfile['terms']
            lenght = Atotsqfile['length']
            print("File loaded successfully.")
        winsound.Beep(1000, 2000)
        
    except Exception as e:
        print(f"❌ Errore durante il caricamento: {e}")
        print("Procedo con il calcolo dall'inizio...")
        atotsq_file = None  # Forza il ricalcolo
        
else:
    print("❌ File non trovato. Procedo con il calcolo dell'ampiezza...")

if not atotsq_file:
    print("Calcolando l'ampiezza totale...")
    A_NR = a_NR * sp.conjugate(rot.D(0, 0, 0, phi, theta1, 0)) * sp.conjugate(rot.D(0, 0, 0, chi, theta2, 0)) * sp.conjugate(rot.D(0, 0, 0, 0, thetal, 0))

    Atot = A_NR + A3 + AR + AR1R2

    totalterms = len(Atot.as_ordered_terms())
    print(f"Numero totale di termini nell'ampiezza Atot: {totalterms}")

    # Sostituisci solo i termini BW con L=0 (non L=1, non L=2)
    W1, W2 = sp.Wild('W1'), sp.Wild('W2')
    Atot = Atot.replace(BWfunc(0, W1, W2), 1)
    
    # Monitora l'uso della memoria
    process = psutil.Process()
    print(f"Memoria utilizzata: {process.memory_info().rss / 1024 / 1024:.1f} MB")
    print(f"Memoria totale: {psutil.virtual_memory().total / 1024 / 1024:.1f} MB")

    # AAcL = (Atot * sp.conjugate(Atot)).xreplace({eta: -1}).xreplace(Leptonic_subs).doit()
    # AAcR = (Atot * sp.conjugate(Atot)).xreplace({eta: 1}).xreplace(Leptonic_subs).doit()

    AAcL = 0
    AAcR = 0
  
    # Quadrato dell'ampiezza totale con chiralità eta = -1
    AAcL = (
        (Atot*sp.conjugate(Atot)).subs({eta: -1, lambda1: -sp.Rational(1,2), lambda2: sp.Rational(1,2)}).xreplace(Leptonic_subs)
    )
    print("="*25)
    print(f"Numero termini per eta=-1, lambda1={-sp.Rational(1,2)}, lambda2={sp.Rational(1,2)}: {len(AAcL.as_ordered_terms())}")
    
    # Quadrato dell'ampiezza totale con chiralità eta = 1
    AAcR = (
        (Atot*sp.conjugate(Atot)).subs({eta: 1, lambda1: sp.Rational(1,2), lambda2: -sp.Rational(1,2)}).xreplace(Leptonic_subs)
    )
    print("="*25)
    print(f"Numero termini per eta=1, lambda1={sp.Rational(1,2)}, lambda2={-sp.Rational(1,2)}: {len(AAcR.as_ordered_terms())}")
    
    # Espande l'espressione svolgendo i prodotti ma non espande le funzioni trigonometriche
    print("\nEspandendo l'espressione finale.")
    Atotsq = sp.expand(AAcL + AAcR)

    # Calcola i termini
    terms_list = Atotsq.as_ordered_terms()
    lenght = len(terms_list)
    print(f"Numero di termini: {lenght}\n")
    print(f"Numero di parametri a_i: {len(atot)}\n")

    # Stampa 5 termini casuali come esempio
    for k in range(0,5):
        rnd = randint(0, lenght - 1)
        print(f"Termine casuale #{rnd}:")
        sp.pprint(terms_list[rnd])
        print()

    # Salva l'espressione totale, la lista contenente i termini e la lunghezza in un file .pkl
    with open(f"WIGAtotsq_terms{lenght}.pkl", 'wb') as f:
        dill.dump({'Atotsq': Atotsq, 'terms': terms_list, 'length': lenght}, f)

    winsound.Beep(1000, 2000)

Verificando la presenza di file salvati...
Founded files. Import of saved data.
File loaded successfully.
